In [2]:
# 导包
import time
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from pyecharts.charts import Bar3D  # 需要安装下, 即: pip install pyecharts
from pyecharts.commons.utils import JsCode
import pyecharts.options as opts

#### 1. 导入数据

In [3]:
sheet_names = ['2015', '2016', '2017', '2018', '会员等级']
sheet_dict = pd.read_excel('../data/sales.xlsx', sheet_name=sheet_names)

#### 2. 数据预处理

In [4]:
for i in sheet_names[:-1]:
    # 1.去除缺失值
    sheet_dict[i] = sheet_dict[i].dropna()
    # 2.过滤出金额大于1的订单
    sheet_dict[i] = sheet_dict[i][sheet_dict[i]['订单金额'] > 1]
    # 3.固定时间节点，以每年的最后一天作为分析节点
    sheet_dict[i]['max_year_date'] = sheet_dict[i]['提交日期'].max()
    sheet_dict[i]['year'] = sheet_dict[i]['提交日期'].dt.year

# 4.合并数据
sheet_merge = pd.concat(list(sheet_dict.values())[:-1], ignore_index=True)
# 5.增加一列，记录该订单与年末的间隔时间
sheet_merge['date_intervals'] = (sheet_merge['max_year_date'] - sheet_merge['提交日期']).dt.days

#### 3. 数据的统计分析

In [5]:
# 1.数据分组
rfm_gb = sheet_merge.groupby(['year', '会员ID'], as_index=False).agg(
    {'date_intervals': 'min', '订单号': 'count', '订单金额': 'sum'})
# 2.重命名列名
rfm_gb.columns = ['year', '会员ID', 'r', 'f', 'm']
# 3.划分区间    r:最近一次购买时间  f:购买次数  m:购买金额
r_bins = [-1, 79, 255, 365]
f_bins = [0, 2, 5, 130]
m_bins = [1, 69, 1199, 206252]
rfm_gb['r_label'] = pd.cut(rfm_gb['r'], bins=r_bins, labels=range(len(r_bins) - 1, 0, -1))
rfm_gb['f_label'] = pd.cut(rfm_gb['f'], bins=f_bins, labels=range(1, len(r_bins)))
rfm_gb['m_label'] = pd.cut(rfm_gb['m'], bins=m_bins, labels=range(1, len(r_bins)))
# 4.拼接评分
rfm_gb['r_label'] = rfm_gb['r_label'].astype(str)
rfm_gb['f_label'] = rfm_gb['f_label'].astype(str)
rfm_gb['m_label'] = rfm_gb['m_label'].astype(str)
rfm_gb['rfm_group'] = rfm_gb['r_label'] + rfm_gb['f_label'] + rfm_gb['m_label']

#### 4. 导出结果

In [6]:
# 可选：将 RFM 结果写入 MySQL（请先设置环境变量 MYSQL_PASSWORD）
import os
from sqlalchemy import create_engine
engine = create_engine(
    "mysql+pymysql://root:" + os.environ.get("MYSQL_PASSWORD", "") + "localhost:3306/itcast"
)
# rfm_gb.to_sql("rfm_table", engine, index=False, if_exists="replace")


148591

#### 5. 数据可视化

In [9]:
# 1. 按照 用户分群类型(rfm_group), 年份分组, 统计会员总数
display_data = rfm_gb.groupby(['rfm_group', 'year'], as_index=False).agg({'会员ID': 'count'})
# 修改列名.
display_data.columns = ['rfm_group', 'year', 'number']
# 修改 rfm_group的类型为 int类型.
display_data['rfm_group'] = display_data['rfm_group'].astype(int)

# 2. 可视化, 显示图形

# 颜色池
range_color = ['#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8', '#ffffbf',
               '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026']

range_max = int(display_data['number'].max())
c = (
    Bar3D()  #设置了一个3D柱形图对象
    .add(
        "",  #图例
        [d.tolist() for d in display_data.values],  #数据
        xaxis3d_opts=opts.Axis3DOpts(type_="category", name='分组名称'),  #x轴数据类型，名称，rfm_group
        yaxis3d_opts=opts.Axis3DOpts(type_="category", name='年份'),  #y轴数据类型，名称，year
        zaxis3d_opts=opts.Axis3DOpts(type_="value", name='会员数量'),  #z轴数据类型，名称，number
    )
    .set_global_opts(  # 全局设置
        visualmap_opts=opts.VisualMapOpts(max_=range_max, range_color=range_color),  #设置颜色，及不同取值对应的颜色
        title_opts=opts.TitleOpts(title="RFM分组结果"),  #设置标题
    )
)
c.render()  # 数据保存到本地的网页中.
# c.render_notebook() #在notebook中显示

'C:\\Users\\Smasun\\PycharmProjects\\PandasProject\\08_综合案例_RFM案例\\render.html'